In [7]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, roc_auc_score
from sklearn.preprocessing import label_binarize

def evaluate_and_plot(models, X_test, y_test, classes):
    """
    Прогоняет модели на тестовой выборке, печатает метрики 
    и рисует Confusion Matrix и усредненную ROC-кривую (One-vs-Rest).
    """
    # Биннаризуем метки для мультиклассового ROC AUC
    y_test_bin = label_binarize(y_test, classes=classes)
    n_classes = y_test_bin.shape[1]
    
    # Подготавливаем холсты для графиков
    fig_cm, axes_cm = plt.subplots(2, 2, figsize=(14, 12))
    axes_cm = axes_cm.flatten()
    
    plt.figure(figsize=(10, 8)) # Холст для ROC-кривых
    
    for idx, (name, model) in enumerate(models.items()):
        print(f"\n{'='*15} Оценка: {name} {'='*15}")
        
        # 1. Текстовые метрики
        y_pred = model.predict(X_test)
        # zero_division=0 чтобы скрипт не падал, если модель проигнорировала какой-то редкий класс
        print(classification_report(y_test, y_pred, zero_division=0)) 
        
        # 2. Confusion Matrix
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes_cm[idx], 
                    xticklabels=classes, yticklabels=classes)
        axes_cm[idx].set_title(f'Матрица ошибок: {name}')
        axes_cm[idx].set_xlabel('Предсказанный класс')
        axes_cm[idx].set_ylabel('Истинный класс')
        
        # 3. ROC AUC (Macro Average One-vs-Rest)
        # Получаем вероятности (у k-NN, RF, LR и нашего SVM с probability=True есть predict_proba)
        y_score = model.predict_proba(X_test)
        
        # Считаем общий ROC AUC score для отчета
        auc_score = roc_auc_score(y_test, y_score, multi_class='ovr', average='macro')
        print(f"ROC AUC Score (Macro OvR): {auc_score:.4f}")
        
        # Вычисляем усредненную кривую для графика
        fpr_grid = np.linspace(0.0, 1.0, 1000)
        mean_tpr = np.zeros_like(fpr_grid)
        
        for i in range(n_classes):
            fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_score[:, i])
            mean_tpr += np.interp(fpr_grid, fpr, tpr)
            
        mean_tpr /= n_classes
        
        # Рисуем
        plt.plot(fpr_grid, mean_tpr, lw=2, label=f'{name} (AUC = {auc_score:.2f})')

    # Настраиваем график матриц ошибок
    fig_cm.tight_layout()
    plt.show() # Если работаешь не в блокноте, а в скрипте, график вылезет в отдельном окне
    
    # Настраиваем финальный вид графика ROC-кривых
    plt.plot([0, 1], [0, 1], 'k--', lw=2) # Диагональная линия случайного угадывания
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate (Macro Average)')
    plt.ylabel('True Positive Rate (Macro Average)')
    plt.title('Сравнение ROC-кривых (Multiclass OvR)')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.show()

# Пример вызова:
# Для начала вытащим все уникальные классы из тренировочной выборки, 
# чтобы передать их в функцию
# import numpy as np
# unique_classes = np.unique(y_train)
# evaluate_and_plot(best_models, X_test_scaled, y_test, unique_classes)